In [ ]:
!pip install transformers datasets torch scikit-learn

In [1]:
import pandas as pd
import torch
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments

In [9]:
df = pd.read_csv("/content/sample_data/imdb_movies_dataset.csv", engine='python', on_bad_lines='skip')
df = df.sample(1000, random_state=42)  # use 1k as the actual data size

# Rename 'Overview' to 'review' as implied by later cells for text input
df.rename(columns={'Overview': 'review'}, inplace=True)

# Create a 'sentiment' column based on 'IMDB_Rating' for binary classification
# Assuming IMDB_Rating >= 7.0 is positive and < 7.0 is negative
df['sentiment'] = df['IMDB_Rating'].apply(lambda x: 'positive' if x >= 7.0 else 'negative')

print(df.head())
print(df['sentiment'].value_counts())

                                           Poster_Link  \
521  https://m.media-amazon.com/images/M/MV5BMjg5OG...   
737  https://m.media-amazon.com/images/M/MV5BMzA2ND...   
740  https://m.media-amazon.com/images/M/MV5BNzMxNT...   
660  https://m.media-amazon.com/images/M/MV5BODllYj...   
411  https://m.media-amazon.com/images/M/MV5BMzJiZD...   

                            Series_Title Released_Year Certificate  Runtime  \
521                 Trois couleurs: Bleu          1993           U   94 min   
737  Captain America: The Winter Soldier          2014          UA  136 min   
740                       Wreck-It Ralph          2012           U  101 min   
660                          The Sandlot          1993           U  101 min   
411                               Gandhi          1982           U  191 min   

                            Genre  IMDB_Rating  \
521         Drama, Music, Mystery          7.9   
737     Action, Adventure, Sci-Fi          7.7   
740  Animation, Adventure,

In [10]:
#Convert labels to numbers
df['label'] = df['sentiment'].map({'positive': 1, 'negative': 0})

In [11]:
#Preprocessing
# Remove null values
df.dropna(inplace=True)

# Use only required columns
df = df[['review', 'label']]

In [12]:
#Data Splitting
train_texts, temp_texts, train_labels, temp_labels = train_test_split(
    df['review'], df['label'], test_size=0.2, random_state=42
)

val_texts, test_texts, val_labels, test_labels = train_test_split(
    temp_texts, temp_labels, test_size=0.5, random_state=42
)

In [13]:
#Tokenization
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

train_encodings = tokenizer(list(train_texts), truncation=True, padding=True, max_length=128)
val_encodings = tokenizer(list(val_texts), truncation=True, padding=True)
test_encodings = tokenizer(list(test_texts), truncation=True, padding=True)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [14]:
#Create Dataset Class
class IMDbDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = list(labels)

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)

train_dataset = IMDbDataset(train_encodings, train_labels)
val_dataset = IMDbDataset(val_encodings, val_labels)
test_dataset = IMDbDataset(test_encodings, test_labels)

In [15]:
#Load BERT Model
model = AutoModelForSequenceClassification.from_pretrained("distilbert-base-uncased", num_labels=2)
#model = AutoModelForSequenceClassification.from_pretrained("bert-base-uncased", num_labels=2)

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [16]:
#Training Setup
training_args = TrainingArguments(
    output_dir="./results",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=1,
    logging_dir="./logs"
)

`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


In [17]:
#Evaluation Function
def compute_metrics(pred):
    labels = pred.label_ids
    preds = pred.predictions.argmax(-1)

    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average='binary')
    acc = accuracy_score(labels, preds)

    return {
        'accuracy': acc,
        'f1': f1,
        'precision': precision,
        'recall': recall
    }

In [18]:
#Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics
)

In [20]:
#Fine Tuning
trainer.train()

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Step,Training Loss


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=72, training_loss=0.0038881926900810665, metrics={'train_runtime': 175.267, 'train_samples_per_second': 3.258, 'train_steps_per_second': 0.411, 'total_flos': 9602592775620.0, 'train_loss': 0.0038881926900810665, 'epoch': 1.0})

In [21]:
#Evaluation
predictions = trainer.predict(test_dataset)
preds = predictions.predictions.argmax(-1)

# Metrics
print("Accuracy:", accuracy_score(test_labels, preds))

precision, recall, f1, _ = precision_recall_fscore_support(test_labels, preds, average='binary')
print("Precision:", precision)
print("Recall:", recall)
print("F1 Score:", f1)

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Accuracy: 1.0
Precision: 1.0
Recall: 1.0
F1 Score: 1.0


In [22]:
#Score Matrix
cm = confusion_matrix(test_labels, preds)
print("Confusion Matrix:\n", cm)

Confusion Matrix:
 [[72]]


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:407: UserWarning: A single label was found in 'y_true' and 'y_pred'. For the confusion matrix to have the correct shape, use the 'labels' parameter to pass all known labels.
  warnings.warn(


In [24]:
#Experiments
#1.Freez DistilBERT Layers
for param in model.distilbert.parameters():
    param.requires_grad = False

In [26]:
#2. Fine-tune last 2 layers
for param in model.distilbert.transformer.layer[-2:].parameters():
    param.requires_grad = True